# MC

This notebook shows how to prgrametically run SWAT with pertubated inputs. It imports from `python_pipeline_scripts`.

## Load environment

In [1]:
import sys; sys.executable

'c:\\Users\\Usuario\\OneDrive - UNIVERSIDAD DE HUELVA\\Granada\\TrabajoFM\\scripts\\Python_Pipeline_SWAT_Pascal\\swat_pipeline\\trabajoFM\\.venv\\Scripts\\python.exe'

In [2]:
from pathlib import Path
import sys
# Add project root to sys.path for imports
sys.path.insert(0, str(Path().resolve().parent))
from python_pipeline_scripts import utils, runner
from python_pipeline_scripts.provenance_report import write_provenance_reports

# Load config from config.yaml
config = utils.load_config(Path().resolve().parent / 'config' / 'config.yaml')

# Enable autoreload in Jupyter for live code updates
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "1")
except NameError:
    pass  # Not in IPython/Jupyter


# this is were this notebook is located and from where relative paths will be resolved
import os
os.getcwd()

'c:\\Users\\Usuario\\OneDrive - UNIVERSIDAD DE HUELVA\\Granada\\TrabajoFM\\scripts\\Python_Pipeline_SWAT_Pascal\\swat_pipeline\\trabajoFM\\notebooks'

## monte carlo runs

In [3]:
# Ensure verbose logging for raster aggregation (even when cached)
from python_pipeline_scripts import utils
from pathlib import Path

cfg = utils.load_config(Path("../config/config.yaml"))
# Force INFO/DEBUG as you prefer
cfg.setdefault("logging", {})["level"] = "INFO"

# Initialize module loggers with this config BEFORE invoking the functions
utils.get_logger("python_pipeline_scripts.raster_agg", config=cfg)
utils.get_logger("python_pipeline_scripts.transforms.soil_chm", config=cfg)

# Now your call will emit lines like:
# "Reprojected (cached) -> ..." and "Clipped (cached) -> ..."


<Logger python_pipeline_scripts.transforms.soil_chm (INFO)>

In [4]:
# Input paths (run this cell before your main pipeline cell)

from pathlib import Path
from python_pipeline_scripts.provenance_report import build_upstream_inputs

def _resolve(p: str, base: Path) -> Path:
    pp = Path(p)
    return pp if pp.is_absolute() else (base / pp).resolve()

# ---------- CHM inputs ----------
script_dir = Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\script DIFFUSE loads - input .chm")
raster_folder = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster\temp_rasters"
hru_zones_fp= r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Archivos de Cesar Ruben Fernandez De Villaran San Juan - swat_cubillas\cubillas_hru\Watershed\Shapes\hru1.shp"
output_gpkg = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.gpkg"

output_gpkg_p = _resolve(output_gpkg, script_dir)
raster_folder_p = _resolve(raster_folder, script_dir)
zones_fp_p = Path(hru_zones_fp)
manifest_file = Path(str(output_gpkg_p) + ".manifest.json")
chm_csv = str(output_gpkg_p).replace(".gpkg", ".csv")

# ---------- Point loads inputs ----------
point_script_dir = Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\script POINT loads - input .dat")
pop_raster_folder = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\fgoerlich-HIPGDAC-ES-cd11f21\HIPGDAC-ES\1970-2021 copy"
subbasins_fp = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\SWaT outputs\Cubillas\shapes cubillas\Sub_basin.shp"
pop_output_gpkg = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.gpkg"

pop_raster_folder_p = _resolve(pop_raster_folder, point_script_dir)
#subbasins_fp_p = _resolve(subbasins_fp, point_script_dir)
pop_output_gpkg_p = _resolve(pop_output_gpkg, point_script_dir)
point_csv = Path(str(pop_output_gpkg_p).replace(".gpkg", ".csv"))

# ---------- SWAT base + outputs ----------
base_txtinout = r"C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1"
realizations_root = r"C:\SWAT\RSWAT\cubillas\mc_realizations"
results_root = r"C:\SWAT\RSWAT\cubillas\mc_results"

base_txtinout_p = Path(base_txtinout)
realizations_root_p = Path(realizations_root); realizations_root_p.mkdir(parents=True, exist_ok=True)
results_root_p = Path(results_root); results_root_p.mkdir(parents=True, exist_ok=True)

# ---------- Upstream inputs for provenance ----------
upstream_chm = build_upstream_inputs(
    raster_folder=raster_folder_p,
    pattern="*_rediam.tif",
    zones_fp=zones_fp_p,
    gpkg_path=output_gpkg_p,
    csv_path=Path(chm_csv),
)
upstream_point = [point_csv]
upstream = list({Path(p) for p in (upstream_chm + upstream_point)})


# ----------- config ------------------------------

cfg = utils.load_config(Path("../config/config.yaml"))


# ---------- Quick sanity prints ----------
print("repo wide config:", cfg)
print("CHM GPKG:", output_gpkg_p)
print("CHM CSV:", chm_csv)
print("Point GPKG:", pop_output_gpkg_p)
print("Point CSV:", point_csv)
print("Base TxtInOut:", base_txtinout_p)
print("Realizations root:", realizations_root_p)
print("Results root:", results_root_p)
print("Manifest (CHM):", manifest_file)


repo wide config: {'logging': {'level': 'INFO', 'log_dir': 'logs', 'file': 'pipeline.log', 'format': '%(asctime)s | %(levelname)s | %(name)s | %(message)s'}, 'paths': {'project_root': '..', 'data_root': 'data', 'outputs_root': 'outputs', 'base_txtinout': 'C:\\SWAT\\RSWAT\\cubillas\\cubillas_set_219_ruben\\cubillas_BASE_set-219\\BASE recreated from arcswat default\\TxtInOut_1', 'swat_executable': 'C:\\SWAT\\ArcSWAT\\swat_64rel.exe'}, 'input_groups': {'baseline': {'description': 'Base input data set (recreated)', 'folder': 'C:\\SWAT\\RSWAT\\cubillas\\cubillas_set_219_ruben\\cubillas_BASE_set-219\\BASE recreated from arcswat default\\TxtInOut_1'}, 'urban_expansion_2030': {'description': 'Urban expansion scenario for 2030', 'folder': 'data/input_groups/urban_expansion_2030'}, 'climate_rcp45': {'description': 'Climate RCP4.5 scenario', 'folder': 'data/input_groups/climate_rcp45'}}, 'reproducibility': {'random_seed': 12345, 'change_sets': [{'name': 'example_change', 'applies_to_group': 'base

## Run input data transformation pipeline (or load from cache)

In [23]:
# Inputs + aggregator with rebuild flags (CHM + Point); logs all decisions.
from pathlib import Path
from python_pipeline_scripts import utils
from python_pipeline_scripts.raster_agg import raster_zonal_aggregation_to_gpkg_and_csv
from python_pipeline_scripts.transforms.soil_chm import read_n_p_means_from_csv_to_df
from python_pipeline_scripts.transforms.point_dat import read_population_by_subbasin_csv_to_df
from python_pipeline_scripts.transforms.point_utils import normalize_point_year_columns
from python_pipeline_scripts.provenance_report import build_upstream_inputs

# Flags: set True to force (re)create CSVs; set False to use cached CSVs if present
REBUILD_CHM_CSV = False
REBUILD_POINT_CSV = False

# Ensure module loggers are initialized with your config so cached actions are printed
utils.get_logger("python_pipeline_scripts.raster_agg", config=cfg)
utils.get_logger("python_pipeline_scripts.transforms.soil_chm", config=cfg)
log = utils.get_logger("notebook.inputs", config=cfg)

def ensure_chm_csv(*, chm_csv: str, raster_folder_p: Path, zones_fp_p: Path, output_gpkg_p: Path, rebuild: bool):
    chm_csv_p = Path(chm_csv)
    if rebuild or not chm_csv_p.exists():
        log.info("CHM CSV missing or rebuild=True; running raster_zonal_aggregation_to_gpkg_and_csv")
        _ = raster_zonal_aggregation_to_gpkg_and_csv( #raster_zonal_aggregation_to_gpkg(
            raster_folder=raster_folder_p,
            zones_fp=zones_fp_p,
            zone_field="HRU_GIS",
            label_field="OBJECTID",
            output_gpkg=output_gpkg_p,
            files_end_with="_rediam.tif",
            stat_operation="mean",
            raster_alias="full_name",
            zone_meaning="HRU",
            overwrite_cache=rebuild,
            write_manifest=True,
            config=cfg,
        )
        log.info("CHM CSV ready: %s", chm_csv_p)
    else:
        log.info("Using cached CHM CSV: %s", chm_csv_p)
    return chm_csv_p

def ensure_point_csv(*, point_csv: Path, pop_raster_folder_p: Path, subbasins_fp: Path, pop_output_gpkg_p: Path, rebuild: bool):
    if rebuild or not point_csv.exists():
        try:
            from python_pipeline_scripts.TEMP_point_dat_insparation import subbasinPopulationAggregationToGPKG
            log.info("Point CSV missing or rebuild=True; building population GPKG/CSV")
            _ = subbasinPopulationAggregationToGPKG(
                raster_folder=str(pop_raster_folder_p),
                sub_basin_fp=str(subbasins_fp),
                zone_field="GRIDCODE",
                output_gpkg=str(pop_output_gpkg_p),
                overwrite_cache=rebuild,
            )
            if not point_csv.exists():
                raise FileNotFoundError(f"Builder ran but CSV still missing: {point_csv}")
            log.info("Point CSV ready: %s", point_csv)
        except Exception as e:
            log.error("Failed to build point CSV automatically: %s", e)
            raise
    else:
        log.info("Using cached Point CSV: %s", point_csv)
    return point_csv

# Ensure CSVs exist (or rebuild if flags set)
chm_csv_p = ensure_chm_csv(
    chm_csv=chm_csv,
    raster_folder_p=raster_folder_p,
    zones_fp_p=zones_fp_p,
    output_gpkg_p=output_gpkg_p,
    rebuild=REBUILD_CHM_CSV,
)
point_csv_p = ensure_point_csv(
    point_csv=point_csv,
    pop_raster_folder_p=pop_raster_folder_p,
    subbasins_fp=subbasins_fp,
    pop_output_gpkg_p=pop_output_gpkg_p,
    rebuild=REBUILD_POINT_CSV,
)

# Upstream inputs for provenance
upstream_chm = build_upstream_inputs(
    raster_folder=raster_folder_p,
    pattern="*_rediam.tif",
    zones_fp=zones_fp_p,
    gpkg_path=output_gpkg_p,
    csv_path=chm_csv_p,
)
upstream_point = [point_csv_p]
upstream = list({Path(p) for p in (upstream_chm + upstream_point)})

# Aggregator: logs will show if cached/rebuilt
aggregator = lambda: {
    "chm": read_n_p_means_from_csv_to_df(
        str(chm_csv_p),
        id_col="HRU_GIS",
        n_col="mean_Nitrogeno_total_porcent_resample_Rediam",
        #n_col="N_prediction_best_according_weight_bias_accuracy_wc_error_resample",
        p_col="mean_Fosforo_mg_100g_P205_rediam",
    ),
    "point": normalize_point_year_columns(
        read_population_by_subbasin_csv_to_df(point_csv_p, id_col="GRIDCODE", sep=";"),
        id_col="GRIDCODE",
        debug=False,
    ),
}


2026-03-13 18:47:50,053 | INFO | notebook.inputs | Using cached CHM CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.csv
2026-03-13 18:47:50,057 | INFO | notebook.inputs | Using cached Point CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.csv


In [24]:
# MC_SPEC: declare transforms and bounds once, choose a mode
MC_SPEC = {
    "mode": "minmax",  # 'mean' | 'random' | 'extreme' | 'minmax'
    "draws": 20,        # only used when mode == 'random'
    "debug": True,

    "transforms": [
        # CHM: compute base (deterministic)
        {"type": "ops", "target": "chm", "name": "compute_base_ops", "ops": [
            {"src": "mean_Nitrogeno_total_porcent_resample_Rediam", "out": "N_total_mg_kg", "op": "mul",
            #{"src": "N_prediction_best_according_weight_bias_accuracy_wc_error_resample", "out": "N_total_mg_kg", "op": "mul",
             #"factor": 10_000.0, "lower": None, "upper": None, "source": "deterministic"},
             "factor": 0, "lower": None, "upper": None, "source": "deterministic"}, 
            {"src": "mean_Fosforo_mg_100g_P205_rediam", "out": "P_element_mg_kg", "op": "mul",
             #"factor": 10.0 * 0.4364, "lower": None, "upper": None, "source": "deterministic"},
             "factor": 0, "lower": None, "upper": None, "source": "deterministic"},
        ]},

        # CHM: split N pool (bounded)
        {"type": "split", "target": "chm", "src": "N_total_mg_kg", "renormalize": True,
         "outputs": [
             {"name": "Soil NO3 [mg/kg]",       "standard": 0.000, "lower": 0.000, "upper": 0.000},
             {"name": "Soil organic N [mg/kg]", "standard": 0.000, "lower": 0.000, "upper": 0.000},
        ]},

        # CHM: derive P_org from N (bounded)
        {"type": "ops", "target": "chm", "name": "derive_p_org_from_n", "ops": [
            {"src": "N_total_mg_kg", "out": "Soil organic P [mg/kg]", "op": "mul",
             "standard": 0.000, "lower": 0.000, "upper": 0.000},
        ]},

        # CHM: split P pool (fixed)
        {"type": "split", "target": "chm", "src": "P_element_mg_kg", "renormalize": True,
         "outputs": [
             {"name": "Soil labile P [mg/kg]", "standard": 0.000, "lower": 0.000, "upper": 0.000},
        ]},

        # CHM: write CHM files
        {"type": "write_chm", "target": "chm", "id_col": "HRU_GIS",
         "label_map": {
             "Soil NO3 [mg/kg]":     "Soil NO3 [mg/kg]",
             "Soil organic N [mg/kg]": "Soil organic N [mg/kg]",
             "Soil labile P [mg/kg]":  "Soil labile P [mg/kg]",
             "Soil organic P [mg/kg]": "Soil organic P [mg/kg]",
        }},

        # POINT: interpolate (optional if CSV already per-year)
        {"type": "interpolate_years_wide", "target": "point", "id_col": "GRIDCODE", "year_start": 1970, "year_end": 2021},

        # POINT: compute timeseries from population and mg/L specs (bounded)
        {"type": "build_point", "target": "point", "id_col": "GRIDCODE",
        "wastewater_lppd": {"standard": 250, "lower": 100, "upper": 400},
         "mgL_values": {
             "ORGNYR": {"standard": 15,  "lower": 5,  "upper": 20},
             "ORGPYR": {"standard": 3,   "lower": 1, "upper": 4},
             "NO3YR":  {"standard": 0,   "lower": 0,   "upper": 0},
             "NH3YR":  {"standard": 25,  "lower": 15,  "upper": 35},
             "NO2YR":  {"standard": 0,   "lower": 0,   "upper": 0},
             "MINPYR": {"standard": 5,   "lower": 1,   "upper": 9},
             "SEDYR":  {"standard": 720, "lower": 540, "upper": 900},
             "CBODYR": {"standard": 220, "lower": 120, "upper": 320},
             "DISOXYR":{"standard": 2.5, "lower": 1, "upper": 4},
             "CHLAYR": {"standard": 0.001, "lower": 0.0, "upper": 0.005},
         },
         "out_columns": ["YEAR","FLOYR","SEDYR","ORGNYR","ORGPYR","NO3YR","NH3YR","NO2YR","MINPYR","CBODYR","DISOXYR","CHLAYR"]},

        # POINT: write rcyr_*.dat
        {"type": "write_point_dat", "target": "point", "id_col": "GRIDCODE",
         "columns_order": ["YEAR","FLOYR","SEDYR","ORGNYR","ORGPYR","NO3YR","NH3YR","NO2YR","MINPYR","CBODYR","DISOXYR","CHLAYR"],
         "start_year": 1970, "end_year": 2021},
    ]
}


In [25]:
from pathlib import Path
from python_pipeline_scripts.spec_runner import run_from_spec
from python_pipeline_scripts.transforms.soil_chm import read_n_p_means_from_csv_to_df
from python_pipeline_scripts.transforms.point_dat import read_population_by_subbasin_csv_to_df
from python_pipeline_scripts.transforms.point_utils import normalize_point_year_columns
from python_pipeline_scripts.provenance_report import summarize_run, realization_report, build_upstream_inputs



results = run_from_spec(
    spec=MC_SPEC,
    aggregator=aggregator,
    base_txtinout=base_txtinout_p,
    realization_root=realizations_root_p,
    results_root=results_root_p,
    link_file_regexes=[r"^[0-9]+\.chm$", r"^rcyr_.*\.dat$"],
    outputs_to_copy=["output.std", "*.rch", "output.sub", "file.cio"] ,
    config=cfg,
    manifest_file=Path(str(output_gpkg_p) + ".manifest.json"),
    run_model=False,
    upstream_inputs=[Path(chm_csv), Path(point_csv)],
)

ok = sum(1 for r in results if r.success)
run_id = results[0].run_id if results else -1
print(f"Created {len(results)} realizations; {ok} succeeded. run_id={run_id}")
for r in results:
    print(f"- {r.name}: id={r.realization_id} run_id={r.run_id} success={r.success} folder={r.folder}")

print("\n=== Monte Carlo run summary ===")
print(summarize_run(run_id))

if results:
    print(f"\n=== Single realization report (id={results[0].realization_id}) ===")
    print(realization_report(results[0].realization_id))


2026-03-13 18:48:03,622 | INFO | python_pipeline_scripts.mc_engine | MC start | N=2 | seed=0
2026-03-13 18:48:03,634 | INFO | python_pipeline_scripts.transforms.soil_chm | Loaded HRU CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.csv | rows=878
2026-03-13 18:48:03,649 | INFO | python_pipeline_scripts.transforms.point_dat | Loaded subbasin CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.csv | rows=17
2026-03-13 18:48:03,750 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_Nitrogeno_total_porcent_resample_Rediam -> N_total_mg_kg | value=0.0 | mean=None lower=None upper=None
2026-03-13 18:48:03,755 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_

[point] normalized years found: [1970, 1981, 1991, 2001, 2011, 2021]
[point] normalized years found: [1970, 1981, 1991, 2001, 2011, 2021]
[point] Total inhabitants across all subbasins (source years):
  year 1970: 17,607
  year 1981: 14,135
  year 1991: 15,264
  year 2001: 15,578
  year 2011: 15,581
[point] Total inhabitants across all subbasins (interpolated years):
  year 1970: 17,607
  year 1971: 17,291
  year 1972: 16,976
  year 1973: 16,660
  year 1974: 16,344
  …
  year 2017: 15,440
  year 2018: 15,416
  year 2019: 15,393
  year 2020: 15,370
  year 2021: 15,346
[point] Total inhabitants across all subbasins (from timeseries input):
  year 1970: 17,607
  year 1971: 17,291
  year 1972: 16,976
  year 1973: 16,660
  year 1974: 16,344
  …
  year 2017: 15,440
  year 2018: 15,416
  year 2019: 15,393
  year 2020: 15,370
  year 2021: 15,346


2026-03-13 18:48:04,653 | INFO | python_pipeline_scripts.transforms.point_dat | Wrote 17 rcyr_*.dat files to C:\SWAT\RSWAT\cubillas\mc_realizations\run000168_real000550_1
2026-03-13 18:48:04,687 | INFO | python_pipeline_scripts.mc_engine | Provenance appended | id=550 | name=run000168_real000550_1 | ledger=C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\provenance\realizations.jsonl
2026-03-13 18:48:04,737 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_Nitrogeno_total_porcent_resample_Rediam -> N_total_mg_kg | value=0.0 | mean=None lower=None upper=None
2026-03-13 18:48:04,740 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_Fosforo_mg_100g_P205_rediam -> P_element_mg_kg | value=0.0 | mean=None lower=None upper=None
2026-03-13 18:48:04,747 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] split N_total_mg_kg -> Soil NO3 [mg/kg] | ra

[point] normalized years found: [1970, 1981, 1991, 2001, 2011, 2021]
[point] normalized years found: [1970, 1981, 1991, 2001, 2011, 2021]
[point] Total inhabitants across all subbasins (source years):
  year 1970: 17,607
  year 1981: 14,135
  year 1991: 15,264
  year 2001: 15,578
  year 2011: 15,581
[point] Total inhabitants across all subbasins (interpolated years):
  year 1970: 17,607
  year 1971: 17,291
  year 1972: 16,976
  year 1973: 16,660
  year 1974: 16,344
  …
  year 2017: 15,440
  year 2018: 15,416
  year 2019: 15,393
  year 2020: 15,370
  year 2021: 15,346
[point] Total inhabitants across all subbasins (from timeseries input):
  year 1970: 17,607
  year 1971: 17,291
  year 1972: 16,976
  year 1973: 16,660
  year 1974: 16,344
  …
  year 2017: 15,440
  year 2018: 15,416
  year 2019: 15,393
  year 2020: 15,370
  year 2021: 15,346


2026-03-13 18:48:05,329 | INFO | python_pipeline_scripts.transforms.point_dat | Wrote 17 rcyr_*.dat files to C:\SWAT\RSWAT\cubillas\mc_realizations\run000168_real000551_2
2026-03-13 18:48:05,342 | INFO | python_pipeline_scripts.mc_engine | Provenance appended | id=551 | name=run000168_real000551_2 | ledger=C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\provenance\realizations.jsonl
2026-03-13 18:48:05,344 | INFO | python_pipeline_scripts.mc_engine | MC finished | run_id=168 | 2/2 succeeded | ledger=C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\provenance\realizations.jsonl | ids=[550, 551]
2026-03-13 18:48:05,345 | INFO | python_pipeline_scripts.mc_engine | Reproduce: find records by 'id' in the ledger and replay transforms with the recorded parameters and seeds.


Created 2 realizations; 2 succeeded. run_id=168
- run000168_real000550_1: id=550 run_id=168 success=True folder=C:\SWAT\RSWAT\cubillas\mc_realizations\run000168_real000550_1
- run000168_real000551_2: id=551 run_id=168 success=True folder=C:\SWAT\RSWAT\cubillas\mc_realizations\run000168_real000551_2

=== Monte Carlo run summary ===
Run 168: realizations=2 ids=[550, 551]
Time span: 2026-03-13T17:48:03.697547+00:00 → 2026-03-13T17:48:04.716868+00:00
Names:
  - run000168_real000550_1
  - run000168_real000551_2
Transforms:
  - transform_init_copy
  - fn
  - ops_choices
  - split_choices
  - transform_interpolate_years_wide
  - point_mgL_choices
Inputs (union):
  - C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Archivos de Cesar Ruben Fernandez De Villaran San Juan - swat_cubillas\cubillas_hru\Watershed\Shapes\hru1.shp
Perturbations:
  - N_total_mg_kg: bounds=unbounded strategy=deterministic
  - P_element_mg_kg: bounds=unbounded strategy=deterministic
  - Soil organic P [mg/kg]: bounds=[0

In [ ]:
#write_provenance_reports(results)

NameError: name 'results' is not defined